# Ejemplo de uso del módulo `utils.plots`

Este notebook demuestra cómo importar y utilizar las funciones de visualización de espacio latente y clustering.

In [ ]:
# Imports necesarios
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Importar funciones de visualización desde el módulo utils.plots
from utils.plots import (
    plot_latent_space_overview,
    plot_base_vs_target_comparison,
    plot_cluster_transition_matrix,
    plot_spatial_comparison_inline,
    plot_spatial_comparisons_all_ssp
)

## 1. Cargar datos de latentes y clustering

Asumiendo que ya tienes cargados:
- `LATENTS`: Diccionario con representaciones latentes por modelo
- `CLUSTERING_RESULTS`: Resultados de clustering
- Variables de configuración: `N_PER_SCENARIO`, `K_CLUSTERS`, `SEED`

In [ ]:
# Ejemplo: Cargar desde archivo pickle (ajusta rutas según tu estructura)
import pickle

# Cargar latentes
with open('data/autoencoder_results/latents.pkl', 'rb') as f:
    LATENTS = pickle.load(f)

# Cargar resultados de clustering
with open('data/autoencoder_results/clustering_results.pkl', 'rb') as f:
    CLUSTERING_RESULTS = pickle.load(f)

# Configuración
MODEL_ORDER = ["AE_128", "AE_64", "AE_32"]  # Ajustar según tus modelos
N_PER_SCENARIO = 1000  # Ajustar según tu dataset
K_CLUSTERS = 5  # Número de clusters
SEED = 42

## 2. Visualización general del espacio latente (PCA)

Vista rápida de cómo se distribuyen los latentes BASE y TARGET en 2D.

In [ ]:
# Visualizar espacio latente para cada modelo
for model_key in MODEL_ORDER:
    plot_latent_space_overview(
        model_key=model_key,
        latents_dict=LATENTS[model_key],
        n_per_scenario=N_PER_SCENARIO,
        seed=SEED,
        figsize=(16, 4)
    )

## 3. Comparación detallada BASE vs TARGET por escenario

Visualización con:
- Clusters coloreados con elipses envolventes
- Centroides BASE y TARGET
- Flechas de desplazamiento y distancias
- Matrices de transición entre clusters

In [ ]:
# Analizar evolución de clusters para un modelo específico
model_key = "AE_128"  # Cambiar según necesidad

transition_results = plot_base_vs_target_comparison(
    model_key=model_key,
    latents_dict=LATENTS[model_key],
    clustering_results=CLUSTERING_RESULTS[model_key],
    n_per_scenario=N_PER_SCENARIO,
    k_clusters=K_CLUSTERS,
    scenarios=["245", "370", "585"],  # Todos los SSPs
    seed=SEED
)

## 4. Análisis de matrices de transición

Los resultados retornados contienen las matrices de transición para análisis posterior.

In [ ]:
# Acceder a matrices de transición por escenario
print("\nMatriz de transición SSP245:")
print(transition_results["245"])

print("\nMatriz de transición SSP585:")
print(transition_results["585"])

# Calcular métricas de estabilidad
stability_245 = np.diag(transition_results["245"].values).sum() / transition_results["245"].values.sum()
stability_585 = np.diag(transition_results["585"].values).sum() / transition_results["585"].values.sum()

print(f"\nEstabilidad de clusters (retención diagonal):")
print(f"  SSP245: {stability_245*100:.1f}%")
print(f"  SSP585: {stability_585*100:.1f}%")

## 5. Loop para procesar todos los modelos

Ejemplo de cómo usar las funciones en un workflow completo.

In [ ]:
# Procesar todos los modelos automáticamente
all_transitions = {}

for model_key in MODEL_ORDER:
    print(f"\n{'='*60}")
    print(f"Procesando modelo: {model_key}")
    print(f"{'='*60}")
    
    # Visualización general
    plot_latent_space_overview(
        model_key=model_key,
        latents_dict=LATENTS[model_key],
        n_per_scenario=N_PER_SCENARIO,
        seed=SEED
    )
    
    # Comparación detallada
    transitions = plot_base_vs_target_comparison(
        model_key=model_key,
        latents_dict=LATENTS[model_key],
        clustering_results=CLUSTERING_RESULTS[model_key],
        n_per_scenario=N_PER_SCENARIO,
        k_clusters=K_CLUSTERS,
        scenarios=["245", "370", "585"],
        seed=SEED
    )
    
    all_transitions[model_key] = transitions

## 6. Exportar resultados

Guardar matrices de transición para análisis posterior.

In [ ]:
# Exportar a CSV
import os

output_dir = "reports/cluster_transitions"
os.makedirs(output_dir, exist_ok=True)

for model_key, transitions in all_transitions.items():
    for ssp, df_transition in transitions.items():
        filepath = os.path.join(output_dir, f"{model_key}_SSP{ssp}_transition.csv")
        df_transition.to_csv(filepath)
        print(f"Guardado: {filepath}")

## 7. Visualización espacial de clusters

Mapas geográficos comparando distribución espacial de clusters BASE vs TARGET.

In [ ]:
# Cargar coordenadas espaciales (lat/lon)
# Ejemplo: desde archivo GeoJSON o CSV
coords_df = pd.read_csv('data/coords/valle_aconcagua_coords.csv')  # Ajustar ruta

# Verificar estructura
print(f"Coordenadas cargadas: {len(coords_df)} puntos")
print(coords_df[['lat', 'lon']].head())

### 7.1 Comparación espacial individual

Mapear un solo par BASE vs TARGET (ejemplo: SSP245).

In [ ]:
# Ejemplo: Comparar clusters BASE vs TARGET para SSP245
model_key = "AE_128"
results = CLUSTERING_RESULTS[model_key]

# Extraer labels BASE y TARGET para SSP245
labels_base = results["labels_base"]
labels_B245 = labels_base[:N_PER_SCENARIO]  # Primera fracción es SSP245
labels_T245 = results["labels_T245"]

# Graficar comparación espacial
plot_spatial_comparison_inline(
    labels_base=labels_B245,
    labels_target=labels_T245,
    coords_df=coords_df,
    title_base="BASE (2020-2029)",
    title_target="TARGET (2090-2100)",
    suptitle=f"SSP245 — {model_key}",
    alpha=0.75
)

### 7.2 Comparación espacial automática para todos los SSPs

Procesar y visualizar todos los escenarios SSP de forma automática.

In [ ]:
# Procesar todos los escenarios SSP para un modelo
model_key = "AE_128"

spatial_stats = plot_spatial_comparisons_all_ssp(
    model_key=model_key,
    clustering_results=CLUSTERING_RESULTS[model_key],
    coords_df=coords_df,
    n_per_scenario=N_PER_SCENARIO,
    scenarios=["245", "370", "585"],
    alpha=0.75
)

# Analizar estadísticas de transición espacial
print("\nEstadísticas de transición espacial:")
for ssp, stats in spatial_stats.items():
    print(f"  SSP{ssp}: {stats['changed']}/{stats['total']} píxeles ({stats['pct_changed']:.1f}%)")